In [66]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import os
# import nest_asyncio
import asyncio
import uvicorn
 

api_key = os.getenv("OPENROUTER_API_KEY")



In [67]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [68]:

llm = ChatOpenAI(
      model="openrouter/free",
       base_url="https://openrouter.ai" + "/api/v1",
       api_key=api_key,
       temperature=0.2,
       max_tokens=4000,
       max_retries=3,
)

In [69]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]}

In [70]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node("chat_node",chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge("chat_node",END)

chatbot = graph.compile(checkpointer=checkpointer)

In [71]:
app = FastAPI()

# Enable CORS so your frontend (running on localhost:3000 or similar) can communicate with Python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # In production, restrict this to your domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


In [72]:
# Define Request Body Schema
class ChatRequest(BaseModel):
    message: str
    thread_id: str = "default-session"

In [73]:


@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    config = {"configurable": {"thread_id": req.thread_id}}
    input_message = HumanMessage(content=req.message)
    
    # Run the graph
    result = chatbot.invoke({"messages": [input_message]}, config)
    
    # Extract the latest assistant message
    reply = result["messages"][-1].content
    return {"reply": reply}

# if __name__ == "__main__":
#     import uvicorn
#     uvicorn.run(app, host="127.0.0.1", port=8000)




In [74]:


# Configure the server
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run the server in your notebook cell
await server.serve()

INFO:     Started server process [26720]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49279 - "OPTIONS /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49279 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49279 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49279 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49279 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:50244 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:61452 - "OPTIONS /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:61452 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:61452 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:63966 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:63550 - "POST /chat HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [26720]
